# Chapter 3: CatBoost — Best-in-Class Categorical Handling

In [ ]:
from catboost import CatBoostClassifier

cat_features = ["contract", "internet", "tech_support"]     # the RAW text columns — no manual encoding needed!
X_cat = df.drop(columns=["churn"])
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_cat, y, test_size=0.25, random_state=42, stratify=y)

cat_model = CatBoostClassifier(
    iterations=300,           # CatBoost's name for n_estimators
    depth=5,                   # CatBoost's name for max_depth
    learning_rate=0.05,
    cat_features=cat_features,     # tell CatBoost exactly which columns are categorical
    verbose=0,
    random_seed=42
)
cat_model.fit(X_train_c, y_train_c)

pred_cat = cat_model.predict_proba(X_test_c)[:, 1]
print("CatBoost ROC-AUC:", round(roc_auc_score(y_test_c, pred_cat), 4))

**This is CatBoost's headline feature, and it's worth appreciating why it matters:** notice that `X_cat` here is the *original* DataFrame with `contract`, `internet`, and `tech_support` still as raw text — no `pd.get_dummies()` one-hot encoding step, unlike the XGBoost and LightGBM examples above. CatBoost handles categorical columns natively and intelligently (using a technique called *ordered target statistics* under the hood, which avoids a subtle leakage risk that naive target-encoding approaches can introduce), which means less manual preprocessing code *and*, for data with many categorical columns or categories with many distinct values, often measurably better performance than one-hot encoding a wide categorical column into dozens of sparse binary columns.

💡 **When this matters in practice:** a customer dataset with `region` (5 values), `plan` (4 values), and `acquisition_channel` (10 values) is exactly the kind of table where reaching for CatBoost first, rather than one-hot-encoding everything for XGBoost/LightGBM, will typically save you both preprocessing effort and model performance.

### Comparing All Three at Once

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values().tail(8).plot.barh(ax=axes[0], title="XGBoost")
pd.Series(lgb_model.feature_importances_, index=X.columns).sort_values().tail(8).plot.barh(ax=axes[1], title="LightGBM")
pd.Series(cat_model.feature_importances_, index=X_cat.columns).sort_values().tail(8).plot.barh(ax=axes[2], title="CatBoost")
plt.tight_layout()
plt.show()

Running all three side by side on the same problem, as this comparison chart does, is standard professional practice for any serious tabular ML project — the "best" library genuinely varies by dataset, and the cost of trying all three (a few extra lines of near-identical code, given how similar their APIs are) is trivial compared to the value of picking the actually-best performer for your specific data.

### Cheat Sheet — CatBoost

| Task | Code |
|---|---|
| Classifier | `CatBoostClassifier(iterations, depth, learning_rate, cat_features=[...])` |
| No encoding needed | Pass raw text columns directly via `cat_features` |
| Silence output | `verbose=0` |

---